# 🚢 Training Pipeline: Baltic Supramax Freight Index (SI) Model
### Smart India Hackathon (SIH 2024) — Vessel Chartering & Bulk Procurement
This interactive notebook trains, evaluates, and forecasts the **Baltic Supramax Index (SI)** for 45,000–60,000 DWT geared bulk carriers.


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

print("Libraries loaded successfully!")


### 1. Load Dataset (`dataset_used_supramax_bdi.csv`)


In [ ]:
df = pd.read_csv('dataset_used_supramax_bdi.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print("Dataset Shape:", df.shape)
df.head(10)


### 2. Time Series Plot & Summary


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['SI'], color='#f59e0b', linewidth=1.5, label='Baltic Supramax Index (SI)')
plt.title('Historical Baltic Supramax Index (SI) Trend', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Index Points')
plt.legend()
plt.show()


### 3. Feature Engineering & Time-Series Split


In [ ]:
for lag in [1, 2, 3, 5, 7, 14, 21, 30]:
    df[f'SI_lag_{lag}'] = df['SI'].shift(lag)

for w in [7, 14, 30]:
    df[f'SI_roll_mean_{w}'] = df['SI'].shift(1).rolling(w).mean()
    df[f'SI_roll_std_{w}']  = df['SI'].shift(1).rolling(w).std()

df['dayofweek'] = df['Date'].dt.dayofweek
df['month'] = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter

data = df.dropna().reset_index(drop=True)
feature_cols = [c for c in data.columns if c not in ['Date', 'SI']]

split_idx = int(len(data) * 0.8)
train_df, test_df = data.iloc[:split_idx], data.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df['SI']
X_test, y_test = test_df[feature_cols], test_df['SI']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


### 4. Train Models & Compare Metrics


In [ ]:
models = {
    'Ridge': Ridge(alpha=10.0),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0)
}

results, predictions = [], {'Date': test_df['Date'].values, 'Actual': y_test.values}

for name, model in models.items():
    if name == 'Ridge':
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    
    predictions[name] = preds
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE (%)': mape})

results_df = pd.DataFrame(results).sort_values('MAE')
print(results_df.to_string(index=False))


### 5. Save Supramax Model


In [ ]:
best_model = models['Ridge']
joblib.dump(best_model, 'SI_best_model.joblib')
pd.DataFrame(predictions).to_csv('test_predictions.csv', index=False)
print("Model saved to 'SI_best_model.joblib'!")
